# Chronos-2 Electricity Forecasting: LoRA Fine-Tuning

## Overview

This notebook implements Parameter-Efficient Fine-Tuning (PEFT) using LoRA (Low-Rank Adaptation) on the pretrained Amazon Chronos-2 model using the hourly Electricity dataset.

The main tasks performed in this notebook are:

- Setting up the environment and loading the prepared dataset splits.
- Configuring the training inputs and validation windows.
- Fine-tuning the Chronos-2 model using LoRA.
- Evaluating the fine-tuned model's forecasting performance across all validation windows.
- Comparing the LoRA fine-tuned model against the zero-shot baseline.
- Saving the fine-tuned adapter weights and evaluation metrics.

## 1. Environment Setup and Pretrained Model Loading

The required Python packages for Chronos forecasting and PEFT are installed.

The pretrained Chronos-2 pipeline is loaded onto the available CUDA device to enable GPU-based training.

In [1]:
!pip install -q chronos-forecasting peft

In [ ]:
import numpy as np
import torch
from chronos import Chronos2Pipeline
!pip install --upgrade torchao==0.18.0 --index-url https://download.pytorch.org/whl/cu130

print("GPU available:", torch.cuda.is_available())

pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cuda"
)

## 2. Loading Prepared Data

Google Drive is mounted to access the dataset splits saved during the data preparation stage. The training, validation, and test sets are loaded from the saved NumPy archive.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

save_dir = "/content/drive/MyDrive/Chronos2_Finetune"

saved = np.load(f"{save_dir}/dataset_splits.npz")

train_data = saved["train"]
val_data = saved["val"]
test_data = saved["test"]

print("Train:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train: (321, 18412)
Validation: (321, 3946)
Test: (321, 3946)


## 3. Training and Validation Data Preparation

The training data is formatted into a list of 1D arrays, as required by the `Chronos2Pipeline`.

To evaluate the model during fine-tuning, validation windows are extracted from the historical context. To reduce validation overhead, a subset of evenly spaced 24-hour validation windows is selected.

In [3]:
# Training inputs: one series per customer
train_inputs = [
    np.asarray(series, dtype=np.float32)
    for series in train_data
]

# Validation: weekly-spaced 24-hour forecast windows
validation_inputs = []

for i in range(len(train_data)):
    history = np.concatenate([train_data[i], val_data[i]])

    for start in range(0, len(val_data[i]) - 24 + 1, 168):
        origin = len(train_data[i]) + start

        window = history[origin - 168 : origin + 24]
        validation_inputs.append(
            np.asarray(window, dtype=np.float32)
        )

print("Training series:", len(train_inputs))
print("Validation windows:", len(validation_inputs))
print("Training shape:", train_inputs[0].shape)
print("Validation window shape:", validation_inputs[0].shape)

Training series: 321
Validation windows: 7704
Training shape: (18412,)
Validation window shape: (192,)


In [4]:
# Select 4 evenly spaced validation windows per customer
# This reduces validation overhead during fine-tuning

validation_inputs_fit = [
    window
    for i in range(0, len(validation_inputs), 24)
    for window in validation_inputs[i:i+24:6]
]

print("Validation windows for training:", len(validation_inputs_fit))

Validation windows for training: 1284


## 4. LoRA Fine-Tuning

The Chronos-2 model is fine-tuned using Low-Rank Adaptation (LoRA) for 500 steps.

Training time and peak GPU memory are tracked, and the fine-tuned adapter weights, alongside the training results, are saved.

In [7]:
import time

torch.cuda.reset_peak_memory_stats()

start_time = time.time()

# Fine-tune Chronos-2 using LoRA
finetuned_pipeline = pipeline.fit(
    inputs=train_inputs,
    validation_inputs=validation_inputs_fit,
    prediction_length=24,
    finetune_mode="lora",
    context_length=168,
    learning_rate=1e-5,
    num_steps=500,
    batch_size=4,
    output_dir=f"{save_dir}/Chronos2_LoRA_Checkpoint"
)

training_time = time.time() - start_time
peak_gpu_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)

print(f"Training time: {training_time / 60:.2f} minutes")
print(f"Peak GPU memory: {peak_gpu_memory:.2f} GB")

Step,Training Loss,Validation Loss
100,2.183531,2.182615
200,2.124667,2.181481
300,2.042798,2.182776
400,2.050417,2.184267
500,2.025126,2.184682


Training time: 2.73 minutes
Peak GPU memory: 1.41 GB


In [8]:
import os

checkpoint_dir = f"{save_dir}/Chronos2_LoRA_Checkpoint"

for root, dirs, files in os.walk(checkpoint_dir):
    print("Directory:", root)
    print("Files:", files)

Directory: /content/drive/MyDrive/Chronos2_Finetune/Chronos2_LoRA_Checkpoint
Files: []
Directory: /content/drive/MyDrive/Chronos2_Finetune/Chronos2_LoRA_Checkpoint/checkpoint-200
Files: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'training_args.bin', 'trainer_state.json']
Directory: /content/drive/MyDrive/Chronos2_Finetune/Chronos2_LoRA_Checkpoint/finetuned-ckpt
Files: ['README.md', 'adapter_model.safetensors', 'adapter_config.json']


In [9]:
import json

training_results = {
    "model": "amazon/chronos-2",
    "finetuning_method": "LoRA",
    "training_steps": 500,
    "batch_size": 4,
    "learning_rate": 1e-5,
    "context_length": 168,
    "prediction_length": 24,
    "training_time_minutes": training_time / 60,
    "peak_gpu_memory_gb": peak_gpu_memory,
    "final_training_loss": 2.025126,
    "final_validation_loss": 2.184682,
    "best_validation_loss": 2.181481,
    "best_step": 200
}

# Save results to Google Drive
with open(f"{save_dir}/lora_training_results.json", "w") as f:
    json.dump(training_results, f, indent=4)

print("Training results saved")

Training results saved


## 5. Validation Evaluation and Comparison

The best-performing LoRA adapter is loaded to evaluate its forecasting accuracy across all validation windows.

Forecasts are generated for the 24-hour horizon using the median (0.5) quantile. These predictions are then evaluated against the actual observations and compared to the zero-shot performance of the base Chronos-2 model.

Performance is assessed using MAE, RMSE, and MASE, alongside a win-rate comparison to quantify the improvements achieved through PEFT.

In [10]:
# Path to the best fine-tuned LoRA adapter
best_checkpoint = (
    "/content/drive/MyDrive/Chronos2_Finetune/"
    "Chronos2_LoRA_Checkpoint/finetuned-ckpt"
)

lora_pipeline = Chronos2Pipeline.from_pretrained(
    best_checkpoint,
    device_map="cuda"
)

print("Best LoRA checkpoint loaded successfully")

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Best LoRA checkpoint loaded successfully


In [11]:
sample_window = validation_inputs[0]

context = torch.tensor(
    sample_window[:168],
    dtype=torch.float32
).reshape(1, 1, 168)

actual = sample_window[168:]

# Generate a 24-hour forecast
forecast = lora_pipeline.predict(
    context,
    prediction_length=24
)

quantiles = lora_pipeline.model.config.chronos_config["quantiles"]
median_index = quantiles.index(0.5)

prediction = forecast[0][0, median_index, :].cpu().numpy()

print("Actual shape:", actual.shape)
print("Prediction shape:", prediction.shape)
print("MAE:", np.mean(np.abs(actual - prediction)))

Actual shape: (24,)
Prediction shape: (24,)
MAE: 2.8924835


In [12]:
# Store actual values and predictions for all validation windows
all_actuals = []
all_predictions = []

for window in validation_inputs:
    # Extract the 168-hour context
    context = torch.tensor(
        window[:168],
        dtype=torch.float32
    ).reshape(1, 1, 168)

    # Extract the actual 24-hour forecast period
    actual = window[168:]

    # Generate the forecast
    forecast = lora_pipeline.predict(
        context,
        prediction_length=24
    )

    # Extract the median forecast
    prediction = forecast[0][0, median_index, :].cpu().numpy()

    all_actuals.extend(actual)
    all_predictions.extend(prediction)

# Calculate validation metrics
all_actuals = np.array(all_actuals)
all_predictions = np.array(all_predictions)

val_mae = np.mean(np.abs(all_actuals - all_predictions))
val_rmse = np.sqrt(np.mean((all_actuals - all_predictions) ** 2))

print("Validation windows:", len(validation_inputs))
print("Validation MAE:", val_mae)
print("Validation RMSE:", val_rmse)

Validation windows: 7704
Validation MAE: 129.14774
Validation RMSE: 1086.8586


In [13]:
np.savez_compressed(
    f"{save_dir}/lora_validation_predictions.npz",
    actuals=all_actuals,
    predictions=all_predictions
)

print("Validation predictions saved!")

Validation predictions saved!


In [14]:
zero_val_actuals = []
zero_val_predictions = []

batch_size = 32

for i in range(0, len(validation_inputs), batch_size):

    batch = validation_inputs[i:i + batch_size]

    # Extract 168-hour contexts and combine into a tensor
    contexts = torch.tensor(
        np.array([window[:168] for window in batch]),
        dtype=torch.float32
    ).unsqueeze(1)

    # Generate 24-hour forecasts using pretrained Chronos-2
    forecasts = pipeline.predict(
        contexts,
        prediction_length=24
    )

    for j, forecast in enumerate(forecasts):
        prediction = forecast[0, median_index, :].cpu().numpy()

        zero_val_predictions.extend(prediction)
        zero_val_actuals.extend(batch[j][168:])

zero_val_actuals = np.array(zero_val_actuals)
zero_val_predictions = np.array(zero_val_predictions)

zero_val_mae = np.mean(
    np.abs(zero_val_actuals - zero_val_predictions)
)

zero_val_rmse = np.sqrt(
    np.mean((zero_val_actuals - zero_val_predictions) ** 2)
)

print("Zero-shot validation MAE:", zero_val_mae)
print("Zero-shot validation RMSE:", zero_val_rmse)

Zero-shot validation MAE: 129.39761
Zero-shot validation RMSE: 1090.3273


In [15]:
zero_errors = np.abs(zero_val_actuals - zero_val_predictions)
lora_errors = np.abs(all_actuals - all_predictions)

print("Zero-shot error percentiles:")
print(np.percentile(zero_errors, [50, 75, 90, 95, 99, 100]))

print("\nLoRA error percentiles:")
print(np.percentile(lora_errors, [50, 75, 90, 95, 99, 100]))

Zero-shot error percentiles:
[2.16690063e+01 6.55463257e+01 1.89259827e+02 3.73651428e+02
 1.50916497e+03 1.00919859e+05]

LoRA error percentiles:
[2.16480980e+01 6.55432434e+01 1.88701813e+02 3.73039429e+02
 1.51254335e+03 1.00793328e+05]


In [17]:
import pandas as pd

actual_by_series = all_actuals.reshape(321, 24, 24)
lora_by_series = all_predictions.reshape(321, 24, 24)
zero_by_series = zero_val_predictions.reshape(321, 24, 24)

# Calculate MAE and RMSE for each customer
series_validation = pd.DataFrame({
    "series_id": [f"T{i+1}" for i in range(321)],
    "LoRA_MAE": np.mean(
        np.abs(actual_by_series - lora_by_series), axis=(1, 2)
    ),
    "ZeroShot_MAE": np.mean(
        np.abs(actual_by_series - zero_by_series), axis=(1, 2)
    ),
    "LoRA_RMSE": np.sqrt(
        np.mean((actual_by_series - lora_by_series) ** 2, axis=(1, 2))
    ),
    "ZeroShot_RMSE": np.sqrt(
        np.mean((actual_by_series - zero_by_series) ** 2, axis=(1, 2))
    )
})

# Display the 10 customers with the highest LoRA RMSE
print(
    series_validation
    .sort_values("LoRA_RMSE", ascending=False)
    .head(10)
    .to_string(index=False)
)

series_id     LoRA_MAE  ZeroShot_MAE    LoRA_RMSE  ZeroShot_RMSE
     T314 10681.777344  10705.457031 17735.019531   17786.568359
     T156  3649.343750   3665.182373  6231.714844    6271.089355
     T237  1673.280762   1673.462280  2212.590576    2214.608154
     T291   870.304688    872.814453  1758.293457    1761.993530
     T316   835.554932    839.988464  1683.156860    1693.364136
     T168   963.782349    964.961609  1574.268555    1575.164307
     T187   713.076416    714.927490  1051.455444    1053.372070
     T309   609.528076    610.278076   966.014771     968.144470
     T315   539.445190    539.575195   906.807495     909.923889
      T90   488.790100    489.997772   894.403259     897.516296


In [18]:
# Calculate percentage MAE improvement for each customer
series_validation["MAE_Improvement_%"] = (
    (series_validation["ZeroShot_MAE"] - series_validation["LoRA_MAE"])
    / series_validation["ZeroShot_MAE"]
) * 100

# Count customers where LoRA performed better
lora_wins = (
    series_validation["LoRA_MAE"] < series_validation["ZeroShot_MAE"]
).sum()

zero_shot_wins = (
    series_validation["LoRA_MAE"] > series_validation["ZeroShot_MAE"]
).sum()

ties = 321 - lora_wins - zero_shot_wins

print("LoRA wins:", lora_wins)
print("Zero-shot wins:", zero_shot_wins)
print("Ties:", ties)

print(
    "Median per-series MAE improvement:",
    series_validation["MAE_Improvement_%"].median()
)

LoRA wins: 238
Zero-shot wins: 83
Ties: 0
Median per-series MAE improvement: 0.0936378687620163


In [19]:
# Calculate the seasonal scaling denominator for each customer
seasonal_scales = np.array([
    np.mean(np.abs(train_data[i][24:] - train_data[i][:-24]))
    for i in range(321)
])

# Calculate LoRA MASE for each customer
series_validation["LoRA_MASE"] = (
    series_validation["LoRA_MAE"] / seasonal_scales
)

# Calculate zero-shot MASE for each customer
series_validation["ZeroShot_MASE"] = (
    series_validation["ZeroShot_MAE"] / seasonal_scales
)

print("Mean LoRA MASE:", series_validation["LoRA_MASE"].mean())
print("Median LoRA MASE:", series_validation["LoRA_MASE"].median())

print("Mean zero-shot MASE:", series_validation["ZeroShot_MASE"].mean())
print("Median zero-shot MASE:", series_validation["ZeroShot_MASE"].median())

Mean LoRA MASE: 0.69542164
Median LoRA MASE: 0.6664908528327942
Mean zero-shot MASE: 0.69620144
Median zero-shot MASE: 0.6683928370475769


## 6. Saving Evaluation Results

The fine-tuned model's predictions and evaluation metrics are saved to Google Drive. These results are stored for subsequent analysis and final benchmarking.

In [20]:
series_validation.to_csv(
    f"{save_dir}/lora_validation_comparison.csv",
    index=False
)

validation_summary = {
    "ZeroShot_MAE": zero_val_mae,
    "LoRA_MAE": val_mae,
    "ZeroShot_RMSE": zero_val_rmse,
    "LoRA_RMSE": val_rmse,
    "ZeroShot_MASE_mean": series_validation["ZeroShot_MASE"].mean(),
    "LoRA_MASE_mean": series_validation["LoRA_MASE"].mean(),
    "ZeroShot_MASE_median": series_validation["ZeroShot_MASE"].median(),
    "LoRA_MASE_median": series_validation["LoRA_MASE"].median()
}

pd.DataFrame([validation_summary]).to_csv(
    f"{save_dir}/lora_validation_summary.csv",
    index=False
)

print("Validation comparison saved")

Validation comparison saved
